<a href="https://colab.research.google.com/github/2008itstudent-rgb/REENA-2/blob/main/%D8%B1%D9%8A%D9%86%D8%A7_%D8%A7%D9%84%D9%83%D9%88%D8%AF_%D9%82%D8%A8%D9%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# ============================================
# 1. تحميل البيانات - المسار الصحيح
# ============================================
def load_music_data(file_path='/content/dataset.csv'):  # 👈 هذا هو المسار الصحيح
    """
    تحميل ملف أغاني Spotify
    """
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        print("✅ تم تحميل بيانات الموسيقى بنجاح!")
        print(f"🎵 عدد الأغاني: {len(df):,}")
        print(f"📋 الأعمدة: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ خطأ في التحميل: {e}")
        return None

# تحميل البيانات (استخدم المسار الصحيح)
df = load_music_data('/content/dataset.csv')  # 👈 هنا كمان غيرنا المسار

# ============================================
# 2. باقي الكود نفسه ما تغير فيه شيء
# ============================================
def prepare_music_data(df):
    """تجهيز إحصائيات سريعة عن البيانات"""
    if df is None:
        return None

    data = {}
    data['total_songs'] = len(df)

    # أشهر الفنانين
    if 'artists' in df.columns:
        data['top_artists'] = df['artists'].value_counts().head(5).to_dict()

    # أنواع الموسيقى
    if 'track_genre' in df.columns:
        data['genres'] = df['track_genre'].unique().tolist()
        data['genre_counts'] = df['track_genre'].value_counts().head(5).to_dict()

    # إحصائيات سريعة
    if 'popularity' in df.columns:
        data['avg_popularity'] = df['popularity'].mean()
        data['max_popularity'] = df['popularity'].max()

    return data

music_stats = prepare_music_data(df)

def search_songs(query, df):
    """البحث عن أغاني حسب الاسم أو الفنان"""
    if df is None:
        return "⚠️ البيانات غير محملة"

    query = str(query).lower().strip()

    # البحث في اسم الأغنية
    if 'track_name' in df.columns:
        songs = df[df['track_name'].str.lower().str.contains(query, na=False)]

        if len(songs) > 0:
            result = f"🎵 **نتائج البحث عن '{query}':**\n\n"
            for idx, song in songs.head(5).iterrows():
                result += f"• {song['track_name']} - {song.get('artists', 'فنان غير معروف')}\n"
                if 'popularity' in song:
                    result += f"  شعبية: {song['popularity']}\n"
            return result
        else:
            return f"لا توجد أغاني تطابق '{query}'"
    return "لا يوجد عمود أسماء الأغاني في البيانات"

def get_songs_by_artist(artist, df):
    """جلب أغاني فنان معين"""
    if df is None or 'artists' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    artist = str(artist).lower().strip()
    songs = df[df['artists'].str.lower().str.contains(artist, na=False)]

    if len(songs) > 0:
        result = f"🎤 **أغاني {artist.title()}:**\n\n"
        for idx, song in songs.head(5).iterrows():
            result += f"• {song.get('track_name', 'غير معروف')}\n"
        return result
    else:
        return f"لا توجد أغاني للفنان '{artist}'"

def get_songs_by_genre(genre, df):
    """جلب أغاني من نوع معين"""
    if df is None or 'track_genre' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    genre = str(genre).lower().strip()
    songs = df[df['track_genre'].str.lower().str.contains(genre, na=False)]

    if len(songs) > 0:
        result = f"🎸 **أغاني من نوع {genre.title()}:**\n\n"
        for idx, song in songs.head(5).iterrows():
            artist = song.get('artists', 'فنان غير معروف')
            track = song.get('track_name', 'أغنية غير معروفة')
            result += f"• {track} - {artist}\n"
        return result
    else:
        return f"لا توجد أغاني من نوع '{genre}'"

def get_top_songs(df, n=5):
    """أشهر الأغاني"""
    if df is None or 'popularity' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    top_songs = df.nlargest(n, 'popularity')
    result = f"🏆 **أشهر {n} أغاني:**\n\n"
    for idx, song in top_songs.iterrows():
        result += f"• {song['track_name']} - {song.get('artists', 'فنان غير معروف')} (شعبية: {song['popularity']})\n"
    return result

def get_music_answer(question):
    global df, music_stats

    if df is None:
        return "⚠️ لم يتم تحميل بيانات الموسيقى. تأكد من رفع ملف CSV."

    question = str(question).strip().lower()

    # أسئلة إحصائية
    if any(word in question for word in ['كم', 'عدد', 'كم أغنية']):
        return f"🎵 إجمالي عدد الأغاني: **{len(df):,}** أغنية"

    elif any(word in question for word in ['أنواع', 'genres']):
        if music_stats and 'genres' in music_stats:
            return f"📊 يوجد **{len(music_stats['genres'])}** نوع موسيقي في البيانات"
        return "لا توجد معلومات عن أنواع الموسيقى"

    elif any(word in question for word in ['أشهر', 'top', 'popular']):
        return get_top_songs(df)

    elif any(word in question for word in ['فنان', 'مغني', 'artist']):
        words = question.split()
        for word in words:
            if len(word) > 3:
                return get_songs_by_artist(word, df)
        return "اكتب اسم الفنان للبحث عن أغانيه"

    elif any(word in question for word in ['نوع', 'genre', 'صنف']):
        words = question.split()
        for word in words:
            if len(word) > 3:
                return get_songs_by_genre(word, df)
        return "اكتب نوع الموسيقى (pop, rock, jazz, etc)"

    else:
        return search_songs(question, df)

def music_chat():
    print("\n" + "="*60)
    print("🎵 **بوت الموسيقى البسيط** 🎵".center(60))
    print("="*60)
    print("\n📌 **الأسئلة المتاحة:**")
    print("• كم عدد الأغاني؟")
    print("• أشهر الأغاني")
    print("• ابحث عن أغنية (اكتب جزء من الاسم)")
    print("• ابحث عن فنان (مثال: أغاني تايلور)")
    print("• ابحث عن نوع موسيقى (مثال: اغاني pop)")
    print("• ما هي أنواع الموسيقى؟")
    print("\n📌 للخروج: quit, exit, bye")
    print("-"*60 + "\n")

    while True:
        user_input = input("🎤 **أنت:** ").strip()

        if user_input.lower() in ['quit', 'exit', 'bye', 'خروج']:
            print("🎵 **البوت:** مع السلامة! استمع لموسيقى جميلة! 👋")
            break

        response = get_music_answer(user_input)
        print(f"🎵 **البوت:** {response}\n")

if __name__ == "__main__":
    if df is not None:
        print(f"\n🎶 **إحصائيات سريعة:**")
        print(f"• عدد الأغاني: {len(df):,}")
        if music_stats and 'genres' in music_stats:
            print(f"• عدد الأنواع: {len(music_stats['genres'])}")
        if 'popularity' in df.columns:
            print(f"• متوسط الشعبية: {df['popularity'].mean():.1f}")
        print("\n" + "="*60)

        music_chat()
    else:
        print("❌ لا يمكن تشغيل البوت بدون بيانات")

✅ تم تحميل بيانات الموسيقى بنجاح!
🎵 عدد الأغاني: 114,000
📋 الأعمدة: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

🎶 **إحصائيات سريعة:**
• عدد الأغاني: 114,000
• عدد الأنواع: 114
• متوسط الشعبية: 33.2


                🎵 **بوت الموسيقى البسيط** 🎵                 

📌 **الأسئلة المتاحة:**
• كم عدد الأغاني؟
• أشهر الأغاني
• ابحث عن أغنية (اكتب جزء من الاسم)
• ابحث عن فنان (مثال: أغاني تايلور)
• ابحث عن نوع موسيقى (مثال: اغاني pop)
• ما هي أنواع الموسيقى؟

📌 للخروج: quit, exit, bye
------------------------------------------------------------

🎤 **أنت:** أشهر الأغاني
🎵 **البوت:** 🏆 **أشهر 5 أغاني:**

• Unholy (feat. Kim Petras) - Sam Smith;Kim Petras (شعبية: 100)
• Unholy (feat. Kim Petras) - Sam Smith;Kim Petras (شعبية: 100)
• Quevedo: Bzrp Music Sessions, Vol. 52